<a href="https://colab.research.google.com/github/adrian-dgx25/E-Commerce-Project-A-B-Testing/blob/main/Project_Data_Enthusiast_A_B_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
from scipy.stats import chisquare
from statsmodels.stats.multitest import multipletests
from typing import Dict, List, Tuple, Optional
import warnings

In [2]:
# ============================================================
# CLASS VALIDATOR
# ============================================================

class ExperimentValidator:

    def __init__(self,
                 srm_threshold: float = 0.001,
                 balance_threshold: float = 0.2,
                 temporal_threshold: float = 0.2):

        self.srm_threshold = srm_threshold
        self.balance_threshold = balance_threshold
        self.temporal_threshold = temporal_threshold


    # --------------------------------------------------------
    # 1. Sample Ratio Mismatch (SRM)
    # --------------------------------------------------------
    def sample_ratio_mismatch_test(
        self,
        df: pd.DataFrame,
        variant_col: str,
        expected_ratio: Optional[Dict[str, float]] = None
    ) -> Dict:

        observed = df[variant_col].value_counts().sort_index()
        total = len(df)
        n_variants = len(observed)

        # Kalau expected_ratio tidak diberikan → otomatis rata
        if expected_ratio is None:
            expected = pd.Series(
                [total / n_variants] * n_variants,
                index=observed.index
            )
        else:
            expected = pd.Series({
                k: v * total for k, v in expected_ratio.items()
            })

        chi2_stat = np.sum((observed - expected) ** 2 / expected)
        df_chi = n_variants - 1
        pvalue = 1 - stats.chi2.cdf(chi2_stat, df_chi)

        has_srm = pvalue < self.srm_threshold

        return {
            'test': 'srm',
            'pvalue': pvalue,
            'has_srm': has_srm,
            'observed': observed.to_dict(),
            'expected': expected.to_dict()
        }


    # --------------------------------------------------------
    # 2. Covariate Balance Check (SMD)
    # --------------------------------------------------------
    def covariate_balance_check(
        self,
        df: pd.DataFrame,
        variant_col: str,
        covariates: List[str]
    ) -> Dict:

        variants = df[variant_col].unique()

        if len(variants) < 2:
            return {'max_smd': 0, 'balance_ok': True}

        results = []

        for col in covariates:
            if col in df.columns:

                v1 = df[df[variant_col] == variants[0]][col]
                v2 = df[df[variant_col] == variants[1]][col]

                if np.issubdtype(df[col].dtype, np.number):

                    pooled_std = np.sqrt((v1.var() + v2.var()) / 2)

                    if pooled_std > 0:
                        smd = abs(v1.mean() - v2.mean()) / pooled_std
                        results.append(smd)

        max_smd = max(results) if results else 0

        return {
            'max_smd': max_smd,
            'balance_ok': max_smd < self.balance_threshold
        }


    # --------------------------------------------------------
    # 3. Temporal Stability Check
    # --------------------------------------------------------
    def temporal_stability_check(
        self,
        df: pd.DataFrame,
        variant_col: str,
        date_col: Optional[str]
    ) -> Dict:

        if date_col is None or date_col not in df.columns:
            return {'is_stable': True, 'max_cv': 0}

        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])

        daily = df.groupby(
            [df[date_col].dt.date, variant_col]
        ).size().unstack(fill_value=0)

        cvs = []

        for col in daily.columns:
            if daily[col].mean() > 0:
                cv = daily[col].std() / daily[col].mean()
                cvs.append(cv)

        max_cv = max(cvs) if cvs else 0

        return {
            'is_stable': max_cv < self.temporal_threshold,
            'max_cv': max_cv
        }


# ============================================================
# VALIDATION FUNCTION PER DATAFRAME
# ============================================================

def validate_variable_data(test_name, df, validator):

    print(f"\n{'='*80}")
    print(f"VALIDATING: {test_name}")
    print(f"{'='*80}")

    # 1. SRM
    srm = validator.sample_ratio_mismatch_test(df, 'variant')

    # 2. Balance
    covs = [c for c in ['device_type', 'browser', 'region'] if c in df.columns]
    balance = validator.covariate_balance_check(df, 'variant', covs)

    # 3. Temporal
    date_col = 'timestamp' if 'timestamp' in df.columns else None
    temporal = validator.temporal_stability_check(df, 'variant', date_col)

    # Status
    srm_status = "PASS" if not srm['has_srm'] else "FAIL"

    print(f"Loaded Rows : {len(df):,}")
    print(f"SRM Test    : {srm_status} (p={srm['pvalue']:.6f})")
    print(f"Balance     : {'OK' if balance['balance_ok'] else 'FAIL'} (SMD={balance['max_smd']:.4f})")
    print(f"Temporal    : {'STABLE' if temporal['is_stable'] else 'UNSTABLE'} (CV={temporal['max_cv']:.4f})")

    return {
        'test': test_name,
        'n': len(df),
        'srm_passed': not srm['has_srm'],
        'balance_ok': balance['balance_ok'],
        'temporal_stable': temporal['is_stable'],
        'overall_valid': (
            not srm['has_srm']
            and balance['balance_ok']
            and temporal['is_stable']
        )
    }


# ============================================================
# RUN ALL DF_1 - DF_5
# ============================================================

def run_all_manual_tests():

    validator = ExperimentValidator()
    results = []

    df_mapping = {
        "Test 1": "df_1",
        "Test 2": "df_2",
        "Test 3": "df_3",
        "Test 4": "df_4",
        "Test 5": "df_5"
    }

    for test_name, df_name in df_mapping.items():

        if df_name in globals():

            df = globals()[df_name]
            res = validate_variable_data(test_name, df, validator)
            results.append(res)

        else:
            print(f"{df_name} tidak ditemukan, dilewati...")

    summary = pd.DataFrame(results)

    # Supaya tidak terpotong
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)

    print("\n\n" + "="*80)
    print("FINAL SUMMARY TABLE")
    print("="*80)
    print(summary)


# ============================================================
# EKSEKUSI
# ============================================================

if __name__ == "__main__":
    run_all_manual_tests()


df_1 tidak ditemukan, dilewati...
df_2 tidak ditemukan, dilewati...
df_3 tidak ditemukan, dilewati...
df_4 tidak ditemukan, dilewati...
df_5 tidak ditemukan, dilewati...


FINAL SUMMARY TABLE
Empty DataFrame
Columns: []
Index: []


**Create Function Expected Ratio**

In [3]:
# Function Auto Expected Ratio

def expected_ratio_uniform(categories):
    n = len(categories)
    return {k: 1 / n for k in categories}

****

In [4]:
# Function Chi Square Test

def chi_square_test(observed, expected_ratio, alpha=0.01):
    """
    observed       : pd.Series (count aktual per group)
    expected_ratio : dict (proporsi harapan per group)
    alpha          : significance level
    """

    # total sample
    total = observed.sum()

    # expected count = ratio * total
    expected = pd.Series({
        k: v * total
        for k, v in expected_ratio.items()
    })

    # chi-square test
    chi_stat, p_value = chisquare(
        f_obs=observed,
        f_exp=expected
    )

    # decision
    decision = "Reject H0" if p_value < alpha else "Fail to Reject H0"

    return {
        "chi_square": chi_stat,
        "p_value": p_value,
        "alpha": alpha,
        "decision": decision
    }

**# EDA Dataset 1: Menu**

In [5]:
# 1. test 1 Menu

df_1 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSm_1J4LTLIg_nsdmmiajMNOHSfEIFoIK9aDKydcyT2Bx9MVIPx-7bv4cuuj3tzNlbXMNcVloLAgmUm/pub?output=csv"
df_1 = pd.read_csv(df_1)
df_1.head()

,session_id,user_id,timestamp,device_type,browser,region,variant,pages_viewed,added_to_cart,bounced,revenue
0,sess_000000,user_001861,2021-03-07,desktop,Safari,Osijek,A_horizontal_menu,2.299070,1,1,2.767615
1,sess_000001,user_001205,2021-03-04,mobile,Chrome,Other,A_horizontal_menu,2.071886,1,0,2.354004
2,sess_000002,user_000685,2021-03-05,desktop,Chrome,Rijeka,A_horizontal_menu,3.159581,1,0,0.547931
3,sess_000003,user_001851,2021-03-07,desktop,Chrome,Split,A_horizontal_menu,2.568082,1,0,5.348691
4,sess_000004,user_000187,2021-03-03,mobile,Chrome,Split,A_horizontal_menu,1.433892,1,0,2.790300


**STEP 1: A/B TESTING FOR DF_1**

In [21]:
# ===========================
# ABTestAnalyzer class
# ===========================

class ABTestAnalyzer:

    def __init__(self, alpha: float = 0.05):
        self.alpha = alpha

    def calculate_sample_size(self,
                              baseline_rate: float,
                              mde: float,
                              alpha: float = 0.05,
                              power: float = 0.80,
                              two_tailed: bool = True) -> int:
        if two_tailed:
            z_alpha = stats.norm.ppf(1 - alpha/2)
        else:
            z_alpha = stats.norm.ppf(1 - alpha)

        z_beta = stats.norm.ppf(power)

        p1 = baseline_rate
        p2 = baseline_rate * (1 + mde)
        p2 = min(p2, 0.999)

        numerator = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))
        denominator = (p2 - p1) ** 2

        n = numerator / denominator

        return int(np.ceil(n))

    def two_sample_ttest(self,
                         control: np.ndarray,
                         treatment: np.ndarray,
                         metric_name: str,
                         equal_var: bool = False) -> Dict:
        control = control[~np.isnan(control)]
        treatment = treatment[~np.isnan(treatment)]

        control_mean = control.mean()
        treatment_mean = treatment.mean()
        control_std = control.std(ddof=1)
        treatment_std = treatment.std(ddof=1)
        n_control = len(control)
        n_treatment = len(treatment)

        statistic, pvalue = stats.ttest_ind(treatment, control, equal_var=equal_var)

        pooled_std = np.sqrt((control_std**2 + treatment_std**2) / 2)
        cohens_d = (treatment_mean - control_mean) / pooled_std if pooled_std > 0 else 0

        se_diff = np.sqrt(control_std**2/n_control + treatment_std**2/n_treatment)

        if not equal_var:
            num = (control_std**2/n_control + treatment_std**2/n_treatment)**2
            denom = ((control_std**2/n_control)**2/(n_control-1) +
                     (treatment_std**2/n_treatment)**2/(n_treatment-1))
            df = num / denom if denom > 0 else n_control + n_treatment - 2
        else:
            df = n_control + n_treatment - 2

        t_crit = stats.t.ppf(1 - self.alpha/2, df)
        diff = treatment_mean - control_mean
        ci_lower = diff - t_crit * se_diff
        ci_upper = diff + t_crit * se_diff

        relative_lift_pct = (diff / control_mean * 100) if control_mean != 0 else 0

        return {
            'metric': metric_name,
            'test_type': 't-test',
            'statistic': float(statistic),
            'pvalue': float(pvalue),
            'significant': bool(pvalue < self.alpha),
            'control_mean': float(control_mean),
            'treatment_mean': float(treatment_mean),
            'control_std': float(control_std),
            'treatment_std': float(treatment_std),
            'absolute_diff': float(diff),
            'relative_lift_pct': float(relative_lift_pct),
            'cohens_d': float(cohens_d),
            'effect_interpretation': self._interpret_cohens_d(cohens_d),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper),
            'n_control': int(n_control),
            'n_treatment': int(n_treatment),
            'degrees_of_freedom': float(df)
        }

    def proportion_test(self,
                        control_successes: int,
                        control_total: int,
                        treatment_successes: int,
                        treatment_total: int,
                        metric_name: str) -> Dict:
        if control_total == 0 or treatment_total == 0:
            raise ValueError(f"control_total ({control_total}) dan treatment_total ({treatment_total}) harus > 0")

        p_control = control_successes / control_total
        p_treatment = treatment_successes / treatment_total

        p_pooled = (control_successes + treatment_successes) / (control_total + treatment_total)

        se = np.sqrt(p_pooled * (1 - p_pooled) * (1/control_total + 1/treatment_total))

        z_stat = (p_treatment - p_control) / se if se > 0 else 0

        pvalue = 2 * (1 - stats.norm.cdf(abs(z_stat)))

        se_diff = np.sqrt(p_control*(1-p_control)/control_total +
                          p_treatment*(1-p_treatment)/treatment_total)
        z_crit = stats.norm.ppf(1 - self.alpha/2)
        diff = p_treatment - p_control
        ci_lower = diff - z_crit * se_diff
        ci_upper = diff + z_crit * se_diff

        relative_lift_pct = (diff / p_control * 100) if p_control > 0 else 0

        return {
            'metric': metric_name,
            'test_type': 'proportion_test',
            'statistic': float(z_stat),
            'pvalue': float(pvalue),
            'significant': bool(pvalue < self.alpha),
            'control_rate': float(p_control),
            'treatment_rate': float(p_treatment),
            'absolute_diff': float(diff),
            'relative_lift_pct': float(relative_lift_pct),
            'ci_lower': float(ci_lower),
            'ci_upper': float(ci_upper),
            'n_control': int(control_total),
            'n_treatment': int(treatment_total)
        }

    def _interpret_cohens_d(self, d: float) -> str:
        ad = abs(d)
        if ad < 0.2:
            return "negligible"
        elif ad < 0.5:
            return "small"
        elif ad < 0.8:
            return "medium"
        else:
            return "large"


# ===========================
# A/B Analysis on df_1
# ===========================

# 1) Ambil dua variant otomatis
variants = df_1['variant'].unique()
print("Variant unik di df_1:", variants)

if len(variants) < 2:
    raise ValueError("Butuh minimal 2 variant untuk A/B test.")

control_name = variants[0]
treatment_name = variants[1]

df_control = df_1[df_1['variant'] == control_name].copy()
df_treat   = df_1[df_1['variant'] == treatment_name].copy()

print(f"Control ({control_name})  : {len(df_control)} rows")
print(f"Treatment ({treatment_name}): {len(df_treat)} rows")

if len(df_control) == 0 or len(df_treat) == 0:
    raise ValueError("Salah satu grup kosong. Cek lagi nama variant dan distribusi datanya.")

# 2) Inisialisasi analyzer
analyzer = ABTestAnalyzer(alpha=0.05)

# 3) T-test untuk metric kontinyu
rev_result = analyzer.two_sample_ttest(
    control=df_control['revenue'].values,
    treatment=df_treat['revenue'].values,
    metric_name="Revenue per session",
    equal_var=False
)

pages_result = analyzer.two_sample_ttest(
    control=df_control['pages_viewed'].values,
    treatment=df_treat['pages_viewed'].values,
    metric_name="Pages viewed per session",
    equal_var=False
)

# 4) Proportion test untuk added_to_cart dan bounce
conv_result = analyzer.proportion_test(
    control_successes=int(df_control['added_to_cart'].sum()),
    control_total=len(df_control),
    treatment_successes=int(df_treat['added_to_cart'].sum()),
    treatment_total=len(df_treat),
    metric_name="Add to cart rate"
)

bounce_result = analyzer.proportion_test(
    control_successes=int(df_control['bounced'].sum()),
    control_total=len(df_control),
    treatment_successes=int(df_treat['bounced'].sum()),
    treatment_total=len(df_treat),
    metric_name="Bounce rate"
)

# 5) Helper untuk print hasil
def print_prop_result(res):
    print(f"\n=== {res['metric']} ===")
    print(f"Control rate   : {res['control_rate']:.4f}")
    print(f"Treatment rate : {res['treatment_rate']:.4f}")
    print(f"Abs diff       : {res['absolute_diff']:.4f}")
    print(f"Lift           : {res['relative_lift_pct']:.2f}%")
    print(f"p-value        : {res['pvalue']:.4g}")
    print(f"CI 95%         : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Significant    : {res['significant']}")

def print_ttest_result(res):
    print(f"\n=== {res['metric']} ===")
    print(f"Control mean    : {res['control_mean']:.4f}")
    print(f"Treatment mean  : {res['treatment_mean']:.4f}")
    print(f"Abs diff        : {res['absolute_diff']:.4f}")
    print(f"Lift            : {res['relative_lift_pct']:.2f}%")
    print(f"p-value         : {res['pvalue']:.4g}")
    print(f"CI 95%          : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Cohen's d       : {res['cohens_d']:.3f} ({res['effect_interpretation']})")
    print(f"Significant     : {res['significant']}")

# 6) Tampilkan ringkasan
print_ttest_result(rev_result)
print_ttest_result(pages_result)
print_prop_result(conv_result)
print_prop_result(bounce_result)

Variant unik di df_1: ['A_horizontal_menu' 'B_dropdown_menu']
Control (A_horizontal_menu)  : 3500 rows
Treatment (B_dropdown_menu): 3500 rows

=== Revenue per session ===
Control mean    : 3.4946
Treatment mean  : 3.1272
Abs diff        : -0.3674
Lift            : -10.51%
p-value         : 1.626e-10
CI 95%          : [-0.4799, -0.2549]
Cohen's d       : -0.153 (negligible)
Significant     : True

=== Pages viewed per session ===
Control mean    : 2.1944
Treatment mean  : 2.1504
Abs diff        : -0.0440
Lift            : -2.01%
p-value         : 0.013
CI 95%          : [-0.0787, -0.0093]
Cohen's d       : -0.059 (negligible)
Significant     : True

=== Add to cart rate ===
Control rate   : 0.9617
Treatment rate : 0.8623
Abs diff       : -0.0994
Lift           : -10.34%
p-value        : 0
CI 95%         : [-0.1125, -0.0864]
Significant    : True

=== Bounce rate ===
Control rate   : 0.4340
Treatment rate : 0.4454
Abs diff       : 0.0114
Lift           : 2.63%
p-value        : 0.3354
CI 

**# EDA Dataset 2: Novelty Slider**

In [6]:
# 2. test 2 Novelty Slider

df_2 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vRLikq1XV_AUah-6PemT_TajvI9sf-g5qs-Hq6RY2zNjnXJ2u29KquizPc8_T91H1FxccuLl_gyyNvX/pub?output=csv"
df_2 = pd.read_csv(df_2)
df_2.head()

,session_id,user_id,timestamp,device_type,browser,region,variant,is_registered,novelty_revenue,products_added_from_novelties
0,sess_000000,user_003937,2021-03-13,desktop,Chrome,Zagreb,A_manual_novelties,1,6.611739,0
1,sess_000001,user_005445,2021-03-18,mobile,Safari,Rijeka,A_manual_novelties,1,2.508166,0
2,sess_000002,user_001121,2021-03-13,mobile,Chrome,Zagreb,A_manual_novelties,1,3.860873,0
3,sess_000003,user_002155,2021-03-19,mobile,Safari,Split,A_manual_novelties,1,5.869096,0
4,sess_000004,user_000349,2021-03-15,tablet,Firefox,Zagreb,A_manual_novelties,0,1.404599,0


In [7]:
# df_2 Dataset Checking

df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   session_id                     16000 non-null  object 
 1   user_id                        16000 non-null  object 
 2   timestamp                      16000 non-null  object 
 3   device_type                    16000 non-null  object 
 4   browser                        16000 non-null  object 
 5   region                         16000 non-null  object 
 6   variant                        16000 non-null  object 
 7   is_registered                  16000 non-null  int64  
 8   novelty_revenue                16000 non-null  float64
 9   products_added_from_novelties  16000 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 1.2+ MB


In [8]:
# Expected Ratio

expected_ratio_uniform(df_2["variant"].unique())

{'A_manual_novelties': 0.5, 'B_personalized_novelties': 0.5}

In [9]:
# Chi Square df_2

observed_df_2 = df_2["variant"].value_counts()
expected_ratio_df_2 = expected_ratio_uniform(df_2["variant"].unique())
chi_square_test(observed_df_2, expected_ratio_df_2)

{'chi_square': np.float64(0.0),
 'p_value': np.float64(1.0),
 'alpha': 0.01,
 'decision': 'Fail to Reject H0'}

**STEP 1: A/B TESTING FOR DF_2**

In [32]:
# =====================================
# Analisis df_2 (Novelties Slider)
# =====================================

print("Variant unik di df_2:", df_2["variant"].unique())
print("Kolom di df_2:", list(df_2.columns))

control_name = "A_manual_novelties"
treatment_name = "B_personalized_novelties"

df_control = df_2[df_2["variant"] == control_name].copy()
df_treat   = df_2[df_2["variant"] == treatment_name].copy()

print(f"\nControl  ({control_name})  : {len(df_control)} rows")
print(f"Treatment({treatment_name}): {len(df_treat)} rows")

if len(df_control) == 0 or len(df_treat) == 0:
    raise ValueError("Salah satu grup kosong. Cek lagi distribusi variant di df_2.")

analyzer = ABTestAnalyzer(alpha=0.05)

# ===========================
# Helper print
# ===========================
def print_prop_result(res, metric_name):
    print(f"\n=== {metric_name} ({treatment_name} vs {control_name}) ===")
    print(f"Control rate   : {res['control_rate']:.4f}")
    print(f"Treatment rate : {res['treatment_rate']:.4f}")
    print(f"Abs diff       : {res['absolute_diff']:.4f}")
    print(f"Lift           : {res['relative_lift_pct']:.2f}%")
    print(f"p-value        : {res['pvalue']:.4g}")
    print(f"CI 95%         : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Significant    : {res['significant']}")

def print_ttest_result(res, metric_name):
    print(f"\n=== {metric_name} ({treatment_name} vs {control_name}) ===")
    print(f"Control mean    : {res['control_mean']:.4f}")
    print(f"Treatment mean  : {res['treatment_mean']:.4f}")
    print(f"Abs diff        : {res['absolute_diff']:.4f}")
    print(f"Lift            : {res['relative_lift_pct']:.2f}%")
    print(f"p-value         : {res['pvalue']:.4g}")
    print(f"CI 95%          : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Cohen's d       : {res['cohens_d']:.3f} ({res['effect_interpretation']})")
    print(f"Significant     : {res['significant']}")


# ===========================
# 1) Metric kontinu: novelty_revenue
# ===========================
if "novelty_revenue" in df_2.columns:
    rev_result = analyzer.two_sample_ttest(
        control=df_control["novelty_revenue"].values,
        treatment=df_treat["novelty_revenue"].values,
        metric_name="Novelty revenue per session",
        equal_var=False
    )
    print_ttest_result(rev_result, "Novelty revenue per session")
else:
    print("\n>> Kolom 'novelty_revenue' tidak ditemukan; skip t-test revenue.")


# ===========================
# 2) Metric binary: is_registered
# ===========================
if "is_registered" in df_2.columns:
    ctrl_succ = int(df_control["is_registered"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["is_registered"].sum())
    trt_tot   = len(df_treat)

    reg_result = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Registration rate"
    )
    print_prop_result(reg_result, "Registration rate")
else:
    print("\n>> Kolom 'is_registered' tidak ditemukan; skip registration rate.")


# ===========================
# 3) Metric binary: products_added_from_novelties
# ===========================
if "products_added_from_novelties" in df_2.columns:
    ctrl_succ = int(df_control["products_added_from_novelties"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["products_added_from_novelties"].sum())
    trt_tot   = len(df_treat)

    add_result = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Products added from novelties rate"
    )
    print_prop_result(add_result, "Products added from novelties rate")
else:
    print("\n>> Kolom 'products_added_from_novelties' tidak ditemukan; skip add-to-cart novelties.")

Variant unik di df_2: ['A_manual_novelties' 'B_personalized_novelties']
Kolom di df_2: ['session_id', 'user_id', 'timestamp', 'device_type', 'browser', 'region', 'variant', 'is_registered', 'novelty_revenue', 'products_added_from_novelties']

Control  (A_manual_novelties)  : 8000 rows
Treatment(B_personalized_novelties): 8000 rows

=== Novelty revenue per session (B_personalized_novelties vs A_manual_novelties) ===
Control mean    : 4.2307
Treatment mean  : 4.4763
Abs diff        : 0.2456
Lift            : 5.81%
p-value         : 8.056e-10
CI 95%          : [0.1673, 0.3240]
Cohen's d       : 0.097 (negligible)
Significant     : True

=== Registration rate (B_personalized_novelties vs A_manual_novelties) ===
Control rate   : 0.4506
Treatment rate : 0.4496
Abs diff       : -0.0010
Lift           : -0.22%
p-value        : 0.8988
CI 95%         : [-0.0164, 0.0144]
Significant    : False

=== Products added from novelties rate (B_personalized_novelties vs A_manual_novelties) ===
Control rat

**# EDA Dataset 3: Product Slider**

In [10]:
# 3. test 3 Product Slider

df_3 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQaL4Q_xQFh_f5PLX9IHn9cbm4N5flmeUdC0X5JwaExIgiMzHIp96o5DSbjMLmsmBgu5uzvebQgpcqK/pub?output=csv"
df_3 = pd.read_csv(df_3)
df_3.head()

,session_id,user_id,timestamp,device_type,browser,region,variant,add_to_cart_rate,slider_interactions,revenue_from_recommendations,products_per_order,avg_product_price
0,sess_000000,user_000567,2021-03-31,mobile,Firefox,Zagreb,A_selected_by_others_only,0,1,3.471560,3.096859,3.284346
1,sess_000001,user_002296,2021-03-28,desktop,Safari,Zagreb,A_selected_by_others_only,0,4,2.661356,3.533376,3.617489
2,sess_000002,user_002093,2021-04-05,desktop,Safari,Rijeka,A_selected_by_others_only,0,2,1.171862,2.662017,4.786812
3,sess_000003,user_003489,2021-04-03,desktop,Safari,Osijek,A_selected_by_others_only,0,2,4.125438,3.888011,5.257324
4,sess_000004,user_000446,2021-03-29,desktop,Chrome,Split,A_selected_by_others_only,0,2,1.884486,2.126060,3.966839


In [11]:
# Expected Ratio

expected_ratio_uniform(df_3["variant"].unique())

{'A_selected_by_others_only': 0.3333333333333333,
 'B_similar_products_top': 0.3333333333333333,
 'C_selected_by_others_top': 0.3333333333333333}

In [12]:
# Chi Square df_3

observed_df_3 = df_3["variant"].value_counts()
expected_ratio_df_3 = expected_ratio_uniform(df_3["variant"].unique())
chi_square_test(observed_df_3, expected_ratio_df_3)

{'chi_square': np.float64(0.0),
 'p_value': np.float64(1.0),
 'alpha': 0.01,
 'decision': 'Fail to Reject H0'}

In [13]:
# df_3 Dataset Checking

df_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   session_id                    18000 non-null  object 
 1   user_id                       18000 non-null  object 
 2   timestamp                     18000 non-null  object 
 3   device_type                   18000 non-null  object 
 4   browser                       18000 non-null  object 
 5   region                        18000 non-null  object 
 6   variant                       18000 non-null  object 
 7   add_to_cart_rate              18000 non-null  int64  
 8   slider_interactions           18000 non-null  int64  
 9   revenue_from_recommendations  18000 non-null  float64
 10  products_per_order            18000 non-null  float64
 11  avg_product_price             18000 non-null  float64
dtypes: float64(3), int64(2), object(7)
memory usage: 1.6+ MB


**STEP 1: A/B TESTING FOR DF_3**

In [33]:
# =====================================
# Analisis df_3: Product Slider (A/B/C)
# =====================================

print("Variant unik di df_3:", df_3["variant"].unique())
print("Kolom di df_3:", list(df_3.columns))

control_name = "A_selected_by_others_only"
treatment_names = [
    "B_similar_products_top",
    "C_selected_by_others_top"
]

analyzer = ABTestAnalyzer(alpha=0.05)

# ===========================
# Helper print
# ===========================
def print_ttest_result(res, metric_name, trt, ctrl):
    print(f"\n=== {metric_name} ({trt} vs {ctrl}) ===")
    print(f"Control mean    : {res['control_mean']:.4f}")
    print(f"Treatment mean  : {res['treatment_mean']:.4f}")
    print(f"Abs diff        : {res['absolute_diff']:.4f}")
    print(f"Lift            : {res['relative_lift_pct']:.2f}%")
    print(f"p-value         : {res['pvalue']:.4g}")
    print(f"CI 95%          : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Cohen's d       : {res['cohens_d']:.3f} ({res['effect_interpretation']})")
    print(f"Significant     : {res['significant']}")

# metric kontinu yang mau diuji (hanya yang ada di df_3)
candidate_metrics = [
    "revenue_from_recommendations",
    "products_per_order",
    "avg_product_price",
    "add_to_cart_rate",
    "slider_interactions",
]

continuous_metrics = [m for m in candidate_metrics if m in df_3.columns]
print("\nMetric kontinu yang akan diuji:", continuous_metrics)

# ===========================
# Loop setiap treatment vs control
# ===========================
for trt in treatment_names:
    print("\n" + "="*70)
    print(f"Comparing {trt} vs {control_name}")
    print("="*70)

    df_control = df_3[df_3["variant"] == control_name].copy()
    df_treat   = df_3[df_3["variant"] == trt].copy()

    print(f"Rows control  ({control_name}): {len(df_control)}")
    print(f"Rows treatment({trt})        : {len(df_treat)}")

    if len(df_control) == 0 or len(df_treat) == 0:
        print(">> SKIP: salah satu grup kosong.")
        continue

    for metric in continuous_metrics:
        res = analyzer.two_sample_ttest(
            control=df_control[metric].values,
            treatment=df_treat[metric].values,
            metric_name=metric,
            equal_var=False
        )
        print_ttest_result(res, metric, trt, control_name)

Variant unik di df_3: ['A_selected_by_others_only' 'B_similar_products_top'
 'C_selected_by_others_top']
Kolom di df_3: ['session_id', 'user_id', 'timestamp', 'device_type', 'browser', 'region', 'variant', 'add_to_cart_rate', 'slider_interactions', 'revenue_from_recommendations', 'products_per_order', 'avg_product_price']

Metric kontinu yang akan diuji: ['revenue_from_recommendations', 'products_per_order', 'avg_product_price', 'add_to_cart_rate', 'slider_interactions']

Comparing B_similar_products_top vs A_selected_by_others_only
Rows control  (A_selected_by_others_only): 6000
Rows treatment(B_similar_products_top)        : 6000

=== revenue_from_recommendations (B_similar_products_top vs A_selected_by_others_only) ===
Control mean    : 4.2971
Treatment mean  : 5.1999
Abs diff        : 0.9027
Lift            : 21.01%
p-value         : 6.477e-59
CI 95%          : [0.7940, 1.0114]
Cohen's d       : 0.297 (small)
Significant     : True

=== products_per_order (B_similar_products_top vs

**# EDA Dataset 4: Reviews**

In [14]:
# 4. test 4 Reviews

df_4 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vT_mLCi9X9wGe69Apmy17q4sp9D_TJuNfpGScspVSlO33v6MO16_IV5VSYwjDehBumdV4ZJHLV_YA8Y/pub?output=csv"
df_4 = pd.read_csv(df_4)
df_4.head()

,session_id,user_id,timestamp,device_type,browser,region,variant,converted,added_to_cart
0,sess_000000,user_001127,2021-04-11,mobile,Chrome,Other,A_no_featured_reviews,0,1
1,sess_000001,user_006812,2021-05-02,mobile,Chrome,Rijeka,A_no_featured_reviews,0,1
2,sess_000002,user_004380,2021-04-27,mobile,Chrome,Zagreb,A_no_featured_reviews,0,1
3,sess_000003,user_009545,2021-04-29,mobile,Chrome,Zagreb,A_no_featured_reviews,0,1
4,sess_000004,user_012102,2021-05-08,mobile,Chrome,Other,A_no_featured_reviews,0,1


In [15]:
# df_4 Dataset Checking

df_4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   session_id     42000 non-null  object
 1   user_id        42000 non-null  object
 2   timestamp      42000 non-null  object
 3   device_type    42000 non-null  object
 4   browser        42000 non-null  object
 5   region         42000 non-null  object
 6   variant        42000 non-null  object
 7   converted      42000 non-null  int64 
 8   added_to_cart  42000 non-null  int64 
dtypes: int64(2), object(7)
memory usage: 2.9+ MB


In [16]:
# Expected Ratio

expected_ratio_uniform(df_4["variant"].unique())

# Chi Square df_4

observed_df_4 = df_4["variant"].value_counts()
expected_ratio_df_4 = expected_ratio_uniform(df_4["variant"].unique())
chi_square_test(observed_df_4, expected_ratio_df_4)

{'chi_square': np.float64(0.0),
 'p_value': np.float64(1.0),
 'alpha': 0.01,
 'decision': 'Fail to Reject H0'}

In [29]:
df_4['variant'].unique()

array(['A_no_featured_reviews', 'B_featured_reviews'], dtype=object)

**STEP 1: A/B TESTING FOR DF_4**

In [34]:
# =====================================
# Analisis df_4: Reviews (A vs B)
# =====================================

print("Variant unik di df_4:", df_4["variant"].unique())
print("Kolom di df_4:", list(df_4.columns))

control_name = "A_no_featured_reviews"
treatment_name = "B_featured_reviews"

df_control = df_4[df_4["variant"] == control_name].copy()
df_treat   = df_4[df_4["variant"] == treatment_name].copy()

print(f"\nControl  ({control_name})  : {len(df_control)} rows")
print(f"Treatment({treatment_name}): {len(df_treat)} rows")

if len(df_control) == 0 or len(df_treat) == 0:
    raise ValueError("Salah satu grup kosong. Cek lagi distribusi variant di df_4.")

analyzer = ABTestAnalyzer(alpha=0.05)

# ===========================
# Helper print
# ===========================
def print_prop_result(res, metric_name):
    print(f"\n=== {metric_name} ({treatment_name} vs {control_name}) ===")
    print(f"Control rate   : {res['control_rate']:.4f}")
    print(f"Treatment rate : {res['treatment_rate']:.4f}")
    print(f"Abs diff       : {res['absolute_diff']:.4f}")
    print(f"Lift           : {res['relative_lift_pct']:.2f}%")
    print(f"p-value        : {res['pvalue']:.4g}")
    print(f"CI 95%         : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Significant    : {res['significant']}")


# ===========================
# 1) Primary: conversion rate (converted)
# ===========================
if "converted" in df_4.columns:
    ctrl_succ = int(df_control["converted"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["converted"].sum())
    trt_tot   = len(df_treat)

    conv_result = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Conversion rate"
    )
    print_prop_result(conv_result, "Conversion rate")
else:
    print("\n>> Kolom 'converted' tidak ditemukan; skip conversion rate.")


# ===========================
# 2) Secondary: add-to-cart rate (added_to_cart)
# ===========================
if "added_to_cart" in df_4.columns:
    ctrl_succ = int(df_control["added_to_cart"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["added_to_cart"].sum())
    trt_tot   = len(df_treat)

    atc_result = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Add to cart rate"
    )
    print_prop_result(atc_result, "Add to cart rate")
else:
    print("\n>> Kolom 'added_to_cart' tidak ditemukan; skip add-to-cart.")

Variant unik di df_4: ['A_no_featured_reviews' 'B_featured_reviews']
Kolom di df_4: ['session_id', 'user_id', 'timestamp', 'device_type', 'browser', 'region', 'variant', 'converted', 'added_to_cart']

Control  (A_no_featured_reviews)  : 21000 rows
Treatment(B_featured_reviews): 21000 rows

=== Conversion rate (B_featured_reviews vs A_no_featured_reviews) ===
Control rate   : 0.1067
Treatment rate : 0.1075
Abs diff       : 0.0009
Lift           : 0.80%
p-value        : 0.7764
CI 95%         : [-0.0051, 0.0068]
Significant    : False

=== Add to cart rate (B_featured_reviews vs A_no_featured_reviews) ===
Control rate   : 0.8268
Treatment rate : 0.8311
Abs diff       : 0.0044
Lift           : 0.53%
p-value        : 0.2332
CI 95%         : [-0.0028, 0.0116]
Significant    : False


**# EDA Dataset 5: Search Engine**

In [17]:
# 5. test 5 Search Engine

df_5 = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQBu1u_osV_ZKMC6Cl67VHuttXeRxEHXgXuu5xcUz1mwCIGwbbRVQXBpX8F0ZtOiWLu3ligU0gBkn7w/pub?output=csv"
df_5 = pd.read_csv(df_5)
df_5.head()

,session_id,user_id,timestamp,device_type,browser,region,variant,avg_revenue_per_visitor,added_to_cart,converted,interacted_with_search
0,sess_000000,user_001321,2021-06-13,desktop,Chrome,Other,A_hybris_search,1.066146,1,0,0
1,sess_000001,user_000722,2021-06-17,mobile,Safari,Split,A_hybris_search,0.817668,1,0,0
2,sess_000002,user_000665,2021-06-14,mobile,Chrome,Zagreb,A_hybris_search,1.341627,1,0,1
3,sess_000003,user_002173,2021-06-14,desktop,Chrome,Other,A_hybris_search,1.389355,1,0,1
4,sess_000004,user_001023,2021-06-13,desktop,Chrome,Other,A_hybris_search,0.285824,1,0,0


In [18]:
# df_5 Dataset Checking

df_5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19000 entries, 0 to 18999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   session_id               19000 non-null  object 
 1   user_id                  19000 non-null  object 
 2   timestamp                19000 non-null  object 
 3   device_type              19000 non-null  object 
 4   browser                  19000 non-null  object 
 5   region                   19000 non-null  object 
 6   variant                  19000 non-null  object 
 7   avg_revenue_per_visitor  19000 non-null  float64
 8   added_to_cart            19000 non-null  int64  
 9   converted                19000 non-null  int64  
 10  interacted_with_search   19000 non-null  int64  
dtypes: float64(1), int64(3), object(7)
memory usage: 1.6+ MB


In [19]:
# Expected Ratio

expected_ratio_uniform(df_5["variant"].unique())

# Chi Square df_5

observed_df_5 = df_5["variant"].value_counts()
expected_ratio_df_5 = expected_ratio_uniform(df_5["variant"].unique())
chi_square_test(observed_df_5, expected_ratio_df_5)

{'chi_square': np.float64(0.0),
 'p_value': np.float64(1.0),
 'alpha': 0.01,
 'decision': 'Fail to Reject H0'}

In [28]:
df_5['variant'].unique()

array(['A_hybris_search', 'B_algolia_search'], dtype=object)

**STEP 1: A/B TESTING FOR DF_5**

In [31]:
# =====================================
# Analisis df_5: Search Engine (Hybris vs Algolia)
# =====================================

print("Variant unik di df_5:", df_5["variant"].unique())
print("Kolom di df_5:", list(df_5.columns))

control_name = "A_hybris_search"
treatment_name = "B_algolia_search"

df_control = df_5[df_5["variant"] == control_name].copy()
df_treat   = df_5[df_5["variant"] == treatment_name].copy()

print(f"\nControl  ({control_name})  : {len(df_control)} rows")
print(f"Treatment({treatment_name}): {len(df_treat)} rows")

if len(df_control) == 0 or len(df_treat) == 0:
    raise ValueError("Salah satu grup kosong. Cek lagi distribusi variant di df_5.")

analyzer = ABTestAnalyzer(alpha=0.05)

# ===========================
# Helper print
# ===========================
def print_prop_result(res, metric_name):
    print(f"\n=== {metric_name} ({treatment_name} vs {control_name}) ===")
    print(f"Control rate   : {res['control_rate']:.4f}")
    print(f"Treatment rate : {res['treatment_rate']:.4f}")
    print(f"Abs diff       : {res['absolute_diff']:.4f}")
    print(f"Lift           : {res['relative_lift_pct']:.2f}%")
    print(f"p-value        : {res['pvalue']:.4g}")
    print(f"CI 95%         : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Significant    : {res['significant']}")

def print_ttest_result(res, metric_name):
    print(f"\n=== {metric_name} ({treatment_name} vs {control_name}) ===")
    print(f"Control mean    : {res['control_mean']:.4f}")
    print(f"Treatment mean  : {res['treatment_mean']:.4f}")
    print(f"Abs diff        : {res['absolute_diff']:.4f}")
    print(f"Lift            : {res['relative_lift_pct']:.2f}%")
    print(f"p-value         : {res['pvalue']:.4g}")
    print(f"CI 95%          : [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"Cohen's d       : {res['cohens_d']:.3f} ({res['effect_interpretation']})")
    print(f"Significant     : {res['significant']}")


# ===========================
# 1) Avg revenue per visitor (t-test)
# ===========================
if "avg_revenue_per_visitor" in df_5.columns:
    rev_result_5 = analyzer.two_sample_ttest(
        control=df_control["avg_revenue_per_visitor"].values,
        treatment=df_treat["avg_revenue_per_visitor"].values,
        metric_name="Avg revenue per visitor",
        equal_var=False
    )
    print_ttest_result(rev_result_5, "Avg revenue per visitor")
else:
    print("\n>> Kolom 'avg_revenue_per_visitor' tidak ditemukan; skip revenue metric.")


# ===========================
# 2) Added to cart rate
# ===========================
if "added_to_cart" in df_5.columns:
    ctrl_succ = int(df_control["added_to_cart"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["added_to_cart"].sum())
    trt_tot   = len(df_treat)

    atc_result_5 = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Add to cart rate"
    )
    print_prop_result(atc_result_5, "Add to cart rate")
else:
    print("\n>> Kolom 'added_to_cart' tidak ditemukan; skip add-to-cart.")


# ===========================
# 3) Conversion rate
# ===========================
if "converted" in df_5.columns:
    ctrl_succ = int(df_control["converted"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["converted"].sum())
    trt_tot   = len(df_treat)

    conv_result_5 = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Conversion rate"
    )
    print_prop_result(conv_result_5, "Conversion rate")
else:
    print("\n>> Kolom 'converted' tidak ditemukan; skip conversion rate.")


# ===========================
# 4) Search interaction rate
# ===========================
if "interacted_with_search" in df_5.columns:
    ctrl_succ = int(df_control["interacted_with_search"].sum())
    ctrl_tot  = len(df_control)
    trt_succ  = int(df_treat["interacted_with_search"].sum())
    trt_tot   = len(df_treat)

    search_result_5 = analyzer.proportion_test(
        control_successes=ctrl_succ,
        control_total=ctrl_tot,
        treatment_successes=trt_succ,
        treatment_total=trt_tot,
        metric_name="Search interaction rate"
    )
    print_prop_result(search_result_5, "Search interaction rate")
else:
    print("\n>> Kolom 'interacted_with_search' tidak ditemukan; skip search interaction.")

Variant unik di df_5: ['A_hybris_search' 'B_algolia_search']
Kolom di df_5: ['session_id', 'user_id', 'timestamp', 'device_type', 'browser', 'region', 'variant', 'avg_revenue_per_visitor', 'added_to_cart', 'converted', 'interacted_with_search']

Control  (A_hybris_search)  : 9500 rows
Treatment(B_algolia_search): 9500 rows

=== Avg revenue per visitor (B_algolia_search vs A_hybris_search) ===
Control mean    : 0.8671
Treatment mean  : 0.8780
Abs diff        : 0.0109
Lift            : 1.26%
p-value         : 0.2887
CI 95%          : [-0.0093, 0.0311]
Cohen's d       : 0.015 (negligible)
Significant     : False

=== Add to cart rate (B_algolia_search vs A_hybris_search) ===
Control rate   : 0.8987
Treatment rate : 0.9123
Abs diff       : 0.0136
Lift           : 1.51%
p-value        : 0.001376
CI 95%         : [0.0053, 0.0219]
Significant    : True

=== Conversion rate (B_algolia_search vs A_hybris_search) ===
Control rate   : 0.0662
Treatment rate : 0.0695
Abs diff       : 0.0033
Lift   